# Lekcia 11 - Protokol medzi agentmi (A2A)


## Nastavenie


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Čo je protokol A2A?

Tento **Agent-to-Agent (A2A) protokol** je otvorený štandard, ktorý umožňuje AI agentom komunikovať,
navzájom sa objavovať a spolupracovať — dokonca aj keď sú postavené na rôznych frameworkoch alebo hosťované
rôznymi službami.

Kľúčové koncepty:

- **Objavovanie** – Agenti zverejňujú *kartu agenta*, ktorá opisuje ich schopnosti, čo
  uľahčuje ostatným agentom (alebo orchestrátorom) nájsť správneho špecialistu pre úlohu.
- **Prenos správ** – Agenti vymieňajú štruktúrované správy cez spoločný protokol, takže
  požiadavka od jedného agenta môže byť pochopená a splnená iným bez ohľadu na jeho vnútornú
  implementáciu.
- **Životný cyklus úlohy** – A2A definuje stavy ako *submitted*, *working*, *completed*, a
  *failed*, čo poskytuje orchestrátorovi plnú viditeľnosť o tom, ako sa delegovaná úloha vyvíja.

V tejto lekcii simulujeme spoluprácu v štýle A2A tým, že prepojíme troch špecializovaných cestovných agentov
do pracovného postupu, v ktorom každý agent prispieva svojimi odbornými znalosťami a odovzdáva výsledky ďalšiemu.


## Vytváranie špecializovaných cestovných agentov


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Spolupráca viacerých agentov pomocou pracovného toku

Prepojíme tri agenty do sekvenčného pracovného toku, ktorý zrkadlí A2A odosielanie správ:

1. **CurrencyExchangeAgent** prijíma požiadavku používateľa a poskytuje odporúčania týkajúce sa meny.
2. **ActivityPlannerAgent** prijíma obohatený kontext a pridáva odporúčania na aktivity.
3. **TravelManagerAgent** spojuje oba vstupy do konečného cestovného zhrnutia.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Pochopenie A2A v produkcii

V produkčnom prostredí protokol A2A umožňuje silné scenáre naprieč službami:

| Capability | Description |
|---|---|
| **Medzirámcová interoperabilita** | Agent vytvorený v jednom rámci môže delegovať úlohy agentovi vytvorenému v akomkoľvek inom rámci kompatibilnom s A2A, čo umožňuje skutočnú interoperabilitu naprieč organizáciami. |
| **Hranice služieb** | Agenti môžu byť umiestnení v samostatných mikroslužbách, cloudových regiónoch alebo dokonca v rôznych organizáciách a pritom spolupracovať bez problémov. |
| **Dynamické objavovanie** | Orchestrátor môže za behu dotazovať registr Agent Card, aby našiel najvhodnejšieho špecialistu pre danú podúlohu. |
| **Streaming & push notifikácie** | A2A podporuje Server-Sent Events (SSE) pre aktualizácie priebehu v reálnom čase a push notifikácie pre dlhšie bežiace úlohy. |

The workflow we built above is a simplified, in-process version of this pattern. In a real
deployment each agent would expose an HTTP endpoint, publish an Agent Card, and communicate
via the A2A JSON-RPC protocol.


## Zhrnutie

V tejto lekcii ste sa naučili:

1. **Čo je protokol A2A** — otvorený štandard pre objavovanie agentov, odosielanie správ,
   a spravovanie úloh.
2. **Ako vytvoriť špecializované agenty** — agenta pre výmenu mien, agenta plánovača aktivít,
   a orchestrátora cestovného manažéra.
3. **Ako prepojiť agentov do pracovného toku** — pomocou `WorkflowBuilder` modelovať sekvenčné
   odovzdávanie správ medzi agentmi.
4. **Ako A2A funguje v produkcii** — umožňujúca spoluprácu naprieč frameworkmi a službami
   s dynamickým objavovaním a priebežnými aktualizáciami.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Vylúčenie zodpovednosti:
Tento dokument bol preložený pomocou služby automatického prekladu založenej na umelej inteligencii [Co-op Translator](https://github.com/Azure/co-op-translator). Aj keď sa snažíme o presnosť, upozorňujeme, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho originálnom jazyku treba považovať za rozhodujúci (autoritatívny) zdroj. Pre dôležité informácie odporúčame profesionálny preklad vykonaný človekom. Nezodpovedáme za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
